Rewriting whole pipeline using Docling integration for langchain

In [35]:
import os

from langchain_docling.loader import DoclingLoader
from langchain_docling.loader import ExportType
from docling.chunking import HybridChunker 

from langchain_core.documents import Document

from hierarchical.postprocessor import ResultPostprocessor


from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct,
    Filter, FieldCondition, MatchValue,
)
from groq import Groq

In [36]:
from pathlib import Path

# Path.cwd() is .../cb-ai/code_files/notebooks_resources/session_4
# .parents[0] -> notebooks_resources
# .parents[1] -> code_files
# .parents[2] -> cb-ai (project root)
PROJECT_ROOT = Path.cwd().parents[2]

SOURCE_PATH = PROJECT_ROOT / "resources" / "Session_4_Resource" / "AtliqAI_HR_Policies.pdf"

# Fail-fast check to ensure the file actually exists
if not SOURCE_PATH.is_file():
    raise FileNotFoundError(f"Target file not found at: {SOURCE_PATH.resolve()}")

# Use as a Path object, or cast to str if the downstream library requires a string
FILE_PATH = str(SOURCE_PATH)

In [37]:
def load_document(source: str):
    loader = DoclingLoader(file_path=source, 
                           chunker=HybridChunker(), # Applies Docling's structure-aware HybridChunker
                            # export_type=ExportType.MARKDOWN 
                           export_type=ExportType.DOC_CHUNKS  # Instructs loader to output chunks, not one giant doc
                           )
    
    document = loader.load()
    # ResultPostprocessor(document).process()
    return document

In [38]:
doc = load_document(FILE_PATH)

The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
[INFO] 2026-09-07 12:55:58,032 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-09-07 12:55:58,039 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-09-07 12:55:58,058 [RapidOCR] download_file.py:60: File exists and is valid: /home/saquib-siddiqui/tensorvault/learning/cb-ai/code_files/cb-ai-venv/lib/python3.14/site-packages/rapidocr/models/PP-OCRv6_det_small.pth
[INFO] 2026-09-07 12:55:58,059 [RapidOCR] main.py:50: Using /home/saquib-siddiqui/tensorvault/learning/cb-ai/code_files/cb-ai-venv/lib/python3.14/site-packages/rapidocr/models/PP-OCRv6_det_small.pth
[INFO] 2026-09-07 12:55:58,360 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-09-07 12:55:58,361 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-09-07 12:

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.


In [39]:
print(doc[:1000])

[Document(metadata={'source': '/home/saquib-siddiqui/tensorvault/learning/cb-ai/resources/Session_4_Resource/AtliqAI_HR_Policies.pdf', 'dl_meta': {'schema_name': 'docling_core.transforms.chunker.DocMeta', 'version': '1.0.0', 'doc_items': [{'self_ref': '#/texts/1', 'parent': {'$ref': '#/body'}, 'children': [], 'content_layer': 'body', 'label': 'text', 'prov': [{'page_no': 1, 'bbox': {'l': 33.301339386, 't': 776.9694969176504, 'r': 548.6781593722308, 'b': 750.5246198306972, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 325]}]}], 'headings': ['AtliqAI HR Policies'], 'origin': {'mimetype': 'application/pdf', 'binary_hash': 257296305029141490, 'filename': 'AtliqAI_HR_Policies.pdf'}}}, page_content='AtliqAI HR Policies\nAtliqAI is committed to building a transparent, inclusive, and high-performance workplace. This document outlines the policies and guidelines that govern employment, conduct, compensation, and well-being at AtliqAI. All employees are expected to read, understand, and adhere 

In [40]:
print(f"Total chunks: {len(doc)}")

Total chunks: 44


In [41]:
# Example: inspect the first chunk
print("Content:", doc[2].page_content)
print("Headings:", doc[2].metadata.get("dl_meta", {}).get("headings", []))

Content: Probation Period
All new employees at AtliqAI are placed on a probation period of 6 months from the date of joining. During this period, either party may terminate the employment with a notice period of 15 days. Performance will be reviewed at the end of the 3rd and 6th month. Successful completion of probation leads to confirmation of employment, which will be communicated in writing by the HR department.
Headings: ['Probation Period']


In [42]:
from langchain_core.documents import Document

def enrich_chunks_with_breadcrumbs(
    chunks: list, 
    document_title: str = "HR Policies"
) -> list[Document]:
    """
    Transforms Docling chunks into LangChain Documents with prepended 
    hierarchical breadcrumbs for semantic search and LLM grounding.
    """
    enriched_docs = []

    for chunk in chunks:
        # Extract headings list (supports both Docling chunk and LangChain Document)
        if hasattr(chunk, "meta") and hasattr(chunk.meta, "headings"):
            headings = chunk.meta.headings or []
            raw_text = chunk.text.strip()
        else:
            headings = chunk.metadata.get("dl_meta", {}).get("headings", [])
            raw_text = chunk.page_content.strip()

        # Build clean breadcrumb trail (e.g., "HR Policies > Leave > Sick Leave")
        full_hierarchy = [document_title] + [h for h in headings if h != document_title]
        breadcrumb_path = " > ".join(full_hierarchy)

        # Clean duplicate heading if chunk text starts with the leaf heading
        leaf_heading = headings[-1] if headings else None
        if leaf_heading and raw_text.startswith(leaf_heading):
            cleaned_text = raw_text[len(leaf_heading):].lstrip(" :\n\r")
        else:
            cleaned_text = raw_text

        # 1. Prepend breadcrumb for embedding and retrieval
        prepended_content = f"Section: {breadcrumb_path}\n\n{cleaned_text}"

        # 2. Preserve structured metadata for filtering and citation formatting
        metadata = {
            "source": document_title,
            "section_path": breadcrumb_path,
            "root_section": full_hierarchy[1] if len(full_hierarchy) > 1 else "General",
            "leaf_section": leaf_heading or "General",
            "raw_content": cleaned_text,  # Keep un-prepended text accessible if needed
        }

        enriched_docs.append(
            Document(page_content=prepended_content, metadata=metadata)
        )

    return enriched_docs

In [43]:
chunks_with_breadcrumbs = enrich_chunks_with_breadcrumbs(doc, document_title="HR Policies")

In [44]:
chunks_with_breadcrumbs

[Document(metadata={'source': 'HR Policies', 'section_path': 'HR Policies > AtliqAI HR Policies', 'root_section': 'AtliqAI HR Policies', 'leaf_section': 'AtliqAI HR Policies', 'raw_content': 'AtliqAI is committed to building a transparent, inclusive, and high-performance workplace. This document outlines the policies and guidelines that govern employment, conduct, compensation, and well-being at AtliqAI. All employees are expected to read, understand, and adhere to these policies from their first day of joining.'}, page_content='Section: HR Policies > AtliqAI HR Policies\n\nAtliqAI is committed to building a transparent, inclusive, and high-performance workplace. This document outlines the policies and guidelines that govern employment, conduct, compensation, and well-being at AtliqAI. All employees are expected to read, understand, and adhere to these policies from their first day of joining.'),
 Document(metadata={'source': 'HR Policies', 'section_path': 'HR Policies > Offer and Join

In [45]:
for chunk in chunks_with_breadcrumbs[:3]:
    print(f"headings   : {chunk.metadata.get('section_path')}")
    print(f"content    : {chunk.page_content[:900]}…")
    print(f"chunk_text : {chunk.metadata.get('raw_content', '')[:900]}…")
    print()

headings   : HR Policies > AtliqAI HR Policies
content    : Section: HR Policies > AtliqAI HR Policies

AtliqAI is committed to building a transparent, inclusive, and high-performance workplace. This document outlines the policies and guidelines that govern employment, conduct, compensation, and well-being at AtliqAI. All employees are expected to read, understand, and adhere to these policies from their first day of joining.…
chunk_text : AtliqAI is committed to building a transparent, inclusive, and high-performance workplace. This document outlines the policies and guidelines that govern employment, conduct, compensation, and well-being at AtliqAI. All employees are expected to read, understand, and adhere to these policies from their first day of joining.…

headings   : HR Policies > Offer and Joining Formalities
content    : Section: HR Policies > Offer and Joining Formalities

Upon acceptance of an offer letter, candidates must complete the joining formalities within the stipulat

## Embedding

In [48]:
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
embedder = SentenceTransformer(EMBEDDING_MODEL)

chunk_texts = [doc.page_content for doc in chunks_with_breadcrumbs]

print(f"Embedding {len(chunk_texts)} chunks …")
embeddings = embedder.encode(chunk_texts, show_progress_bar=True)

print(f"Shape: {embeddings.shape}")   # → (N, 384)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding 44 chunks …


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Shape: (44, 384)


## Indexing

In [49]:
import os
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct,
    Filter,
    FieldCondition,
    MatchValue,
)

# Configuration
QDRANT_URL = "http://localhost:6333"
COLLECTION_NAME = "hr_docs"
RESET_COLLECTION = True  # Set to False to preserve existing points and schema

# Initialize remote client
client = QdrantClient(url=QDRANT_URL, timeout=10)

# 1. Health-check / verify connection to server
try:
    client.get_collections()
except Exception as exc:
    raise ConnectionError(
        f"Unable to reach Qdrant server at '{QDRANT_URL}'. "
        "Ensure the Qdrant container/service is running."
    ) from exc

# 2. Extract and validate vector dimension
try:
    DIM = embedder.get_embedding_dimension()
except AttributeError:
    # Fallback if using LangChain HuggingFaceEmbeddings / client instance
    DIM = getattr(getattr(embedder, "client", None), "get_sentence_embedding_dimension", lambda: None)()
    if DIM is None and hasattr(embedder, "embed_query"):
        DIM = len(embedder.embed_query("dimension_check"))

if not DIM:
    raise ValueError("Could not determine embedding vector dimensions from 'embedder'.")

# 3. Handle existing collection
if client.collection_exists(COLLECTION_NAME):
    if RESET_COLLECTION:
        client.delete_collection(COLLECTION_NAME)
        print(f"Deleted existing collection: '{COLLECTION_NAME}'.")
    else:
        print(f"Collection '{COLLECTION_NAME}' already exists. Skipping recreation.")

# 4. Create collection if it does not exist
if not client.collection_exists(COLLECTION_NAME):
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(
            size=DIM,
            distance=Distance.COSINE,
        ),
    )
    print(f"Collection '{COLLECTION_NAME}' created with dimension {DIM}.")

Deleted existing collection: 'hr_docs'.
Collection 'hr_docs' created with dimension 384.


In [51]:
chunks_with_breadcrumbs

[Document(metadata={'source': 'HR Policies', 'section_path': 'HR Policies > AtliqAI HR Policies', 'root_section': 'AtliqAI HR Policies', 'leaf_section': 'AtliqAI HR Policies', 'raw_content': 'AtliqAI is committed to building a transparent, inclusive, and high-performance workplace. This document outlines the policies and guidelines that govern employment, conduct, compensation, and well-being at AtliqAI. All employees are expected to read, understand, and adhere to these policies from their first day of joining.'}, page_content='Section: HR Policies > AtliqAI HR Policies\n\nAtliqAI is committed to building a transparent, inclusive, and high-performance workplace. This document outlines the policies and guidelines that govern employment, conduct, compensation, and well-being at AtliqAI. All employees are expected to read, understand, and adhere to these policies from their first day of joining.'),
 Document(metadata={'source': 'HR Policies', 'section_path': 'HR Policies > Offer and Join

In [ ]:
for chunk in chunks_with_breadcrumbs[:3]:
    print(f"headings   : {chunk.metadata.get('section_path')}")
    print(f"content    : {chunk.page_content[:900]}…")
    print(f"chunk_text : {chunk.metadata.get('raw_content', '')[:900]}…")
    print()

In [56]:
# Creating Points

points = [
    PointStruct(
        id=idx,
        vector=embedding.tolist(),
        payload={
            "headings":   chunk.metadata.get('section_path'),   # stored as a JSON array
            "content":    chunk.page_content,
            "chunk_text": chunk.metadata.get('raw_content', ''),
        },
    )
    for idx, (chunk, embedding) in enumerate(zip(chunks_with_breadcrumbs, embeddings))
]

result = client.upsert(
    collection_name=COLLECTION_NAME,
    points=points,
    wait=True,
)
print(f"Indexed {len(points)} points — status: {result.status}")

Indexed 44 points — status: completed


In [57]:
info = client.get_collection(COLLECTION_NAME)
print(f"Points     : {info.points_count}")
print(f"Dimensions : {info.config.params.vectors.size}")

Points     : 44
Dimensions : 384


## Retrieval

In [58]:
def retrieve(
    query: str,
    top_k: int = 5
) -> list[dict]:
    """
    Embed the query and return the top-k most similar chunks.

    Args:
        query          : User's question.
        top_k          : Number of chunks to return.
        section_filter : Optional H2 heading to restrict the search scope.
    """
    query_vector = embedder.encode(query).tolist()

    hits = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        limit=top_k,
        with_payload=True,
    )

    return [{**hit.payload, "score": round(hit.score, 4)} for hit in hits.points]

In [59]:
results = retrieve("What is the paternity leave policy?", top_k=3)
for r in results:
    print(f"[{r['score']}]  {r['headings']}")
    print(f"  {r['content'][:200]}…\n")

[0.6967]  HR Policies > Paternity Leave
  Section: HR Policies > Paternity Leave

Male employees and non-birthing partners are entitled to 10 working days of paid paternity leave, to be availed within 6 months of the child's birth or adoption…

[0.463]  HR Policies > Maternity Leave
  Section: HR Policies > Maternity Leave

Female employees who have completed at least 6 months of continuous service are entitled to 26 weeks of paid maternity leave for the first two live births. For …

[0.3458]  HR Policies > Health Insurance
  Section: HR Policies > Health Insurance

All full-time employees and their immediate dependents (spouse and up to 2 children) are covered under a group health insurance policy with a sum insured of ₹5…



## Developing RAG pipeline

In [60]:
SYSTEM_PROMPT = """You are a important HR assistant.
Answer the user's question using ONLY the context provided below.
If the context does not contain enough information, say so — do not make things up.
Always cite the section name when referencing specific information."""

In [61]:
def build_context(retrieved_chunks: list[dict]) -> str:
    parts = []
    for i, chunk in enumerate(retrieved_chunks, 1):
        parts.append(f"[Source {i}]\n{chunk['content']}")
    return "\n\n---\n\n".join(parts)

In [62]:
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

In [63]:
from groq import Groq

groq_client = Groq()   # Reads GROQ_API_KEY from environment automatically
GROQ_MODEL  = "openai/gpt-oss-20b"

def rag(query: str, top_k: int = 5):
    """
    End-to-end RAG pipeline:
      1. Retrieve relevant chunks from Qdrant
      2. Format them as a context block
      3. Send context + query to Groq and return the answer
    """
    # Step 1 — Retrieve
    chunks = retrieve(query, top_k=top_k)
    if not chunks:
        return "No relevant content found in the document."

    # Step 2 — Build context
    context = build_context(chunks)

    # Step 3 — Generate
    user_message = f"Context:\n{context}\n\nQuestion: {query}"

    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_message},
        ],
        temperature=0.2,   # Low = factual;  High = creative
    )
    return response.choices[0].message.content, context

In [64]:
answer, context = rag("How many maternity leaves am I entitled to?")
print(answer)
print(f"{250*'='}")
print(f"\n\nSOURCES:\n {context}")

**Answer (based on the HR policies provided)**  

- **Eligibility**: The policy applies to *female employees* who have completed at least 6 months of continuous service.  
- **Entitlement per birth**:  
  - **First and second live births**: **26 weeks** of paid maternity leave each.  
  - **Third child and onward**: **12 weeks** of paid maternity leave each.  

So, for each child you are entitled to one maternity leave period—26 weeks for your first two children, and 12 weeks for any subsequent children. (If you are not a female employee, the policy does not provide maternity leave.)


SOURCES:
 [Source 1]
Section: HR Policies > Maternity Leave

Female employees who have completed at least 6 months of continuous service are entitled to 26 weeks of paid maternity leave for the first two live births. For the third child onwards, the entitlement is 12 weeks. Maternity leave can begin 8 weeks before the expected delivery date. The employee must submit a medical certificate indicating the e